Assignment 3:

Step 1: List the transaction history of Product table.

Step 2: Restore Product table to a state before the merge transaction was executed.

Step 3: Check the history again after the Restore command. How does it affect the table history?

Step 1: List the transaction history of Product table.

In [0]:
%sql
SELECT * FROM sys.fn_dblog(NULL, NULL) -- For active transaction log
    WHERE AllocUnitName LIKE '%Product%';

OR

In [0]:
%sql
if Change Data Capture (CDC) is enabled:
    SELECT * FROM cdc.fn_cdc_get_all_changes_dbo_Product(sys.fn_cdc_get_min_lsn('dbo_Product'), sys.fn_cdc_get_max_lsn(), 'all');

OR

In [0]:
%sql
SELECT
    [Current LSN],
    Context,
    [Transaction Name],
    Operation,
    [Transaction ID],
    Description
FROM
    fn_dblog(NULL, NULL)
WHERE
    [Transaction Name] = 'ProductTableTransaction' OR -- Assuming a transaction name was used
    AllocUnitName LIKE '%Product%' -- To check log entries related to the table

Step 2: Restore Product table to a state before the merge transaction was executed.

In [0]:
%sql
-- Restore full database backup
RESTORE DATABASE YourDatabaseName FROM DISK = 'C:\YourBackupFile.bak' WITH NORECOVERY;

-- Restore differential backup (if applicable)
RESTORE DATABASE YourDatabaseName FROM DISK = 'C:\YourDiffBackup.bak' WITH NORECOVERY;

-- Restore transaction log backups up to the LSN just before the merge transaction
RESTORE LOG YourDatabaseName FROM DISK = 'C:\YourLogBackup1.bak' WITH NORECOVERY;
RESTORE LOG YourDatabaseName FROM DISK = 'C:\YourLogBackup2.bak' WITH STOPAT = 'YYYY-MM-DD HH:MM:SS', RECOVERY; 

Step 3: Checking the history after the restore command and its effect on table history

After restoring the table, its data will reflect the state before the merge transaction. 
The transaction log will also reflect this restoration operation. 
New entries related to the restore will be added to the log, but previous entries prior to the restore, including the merge transaction, might not be readily available for direct querying using fn_dblog if the log has been truncated or overwritten. 

To ensure a complete and auditable history of changes (including restorations), implementing a robust auditing solution with a dedicated audit table or using SQL Server audit features is recommended.